# Material-World Synthetic Data Pilot

This notebook loads the local `Qwen/Qwen3.5-9B` checkpoint and uses it as a controlled generator/editor for a small material-world synthetic data pilot.

The lower cells rehearse the intended pipeline in miniature: load curated material/object ontologies, generate structured material cards, plan object-material scenarios, render natural prose in varied styles, run a lightweight check pass, and optionally export pilot artifacts.

Important: run the first cell before importing `torch` or `transformers`. If the kernel has already imported either package, restart the kernel first so `CUDA_VISIBLE_DEVICES=3,4` takes effect. Inside Python, physical GPUs 3 and 4 will appear as logical `cuda:0` and `cuda:1`.


In [2]:
import os
import sys
from pathlib import Path

# Set this before importing torch/transformers. Restart the kernel if torch was already imported.
os.environ["CUDA_VISIBLE_DEVICES"] = "3,4"
os.environ.setdefault("HF_HOME", "/SSD2/huggingface")

MODEL_DIR = Path("/SSD2/models/Qwen3.5-9B")
assert MODEL_DIR.exists(), f"Missing model directory: {MODEL_DIR}"
assert (MODEL_DIR / "config.json").exists(), f"Missing config.json under: {MODEL_DIR}"

print(f"MODEL_DIR = {MODEL_DIR}")
print(f"HF_HOME   = {os.environ['HF_HOME']}")
print(f"CUDA_VISIBLE_DEVICES = {os.environ['CUDA_VISIBLE_DEVICES']}")
if "torch" in sys.modules or "transformers" in sys.modules:
    print("WARNING: torch/transformers was already imported. Restart the kernel for GPU 3/4 masking to be reliable.")


MODEL_DIR = /SSD2/models/Qwen3.5-9B
HF_HOME   = /SSD2/huggingface
CUDA_VISIBLE_DEVICES = 3,4


## Version Check

Qwen3.5 is newer than the Transformers version used by some of the BabyLM environments. If the check below warns, run the install cell once, then restart the kernel.


In [3]:
from importlib.metadata import PackageNotFoundError, version


def package_version(name: str) -> str | None:
    try:
        return version(name)
    except PackageNotFoundError:
        return None

versions = {
    "torch": package_version("torch"),
    "transformers": package_version("transformers"),
    "huggingface_hub": package_version("huggingface_hub"),
    "accelerate": package_version("accelerate"),
    "safetensors": package_version("safetensors"),
}
for name, value in versions.items():
    print(f"{name:16s}: {value}")

transformers_version = versions["transformers"] or "0"
hub_version = versions["huggingface_hub"] or "0"

needs_update = False
if transformers_version < "4.57":
    print("WARNING: Qwen3.5 may require a newer Transformers build than this kernel has.")
    needs_update = True
if hub_version >= "1." and transformers_version.startswith("4."):
    print(" WARNING: Transformers 4.x expects huggingface_hub<1.0. Downgrade hub or upgrade Transformers.")
    needs_update = True

if needs_update:
    print("Run the next cell, then restart the kernel.")
else:
    print(" Version check looks plausible. Continuing should work if the model class is available.")


torch           : 2.12.0+cu126
transformers    : 5.13.0.dev0
huggingface_hub : 1.19.0
accelerate      : 1.14.0
safetensors     : 0.8.0
 Version check looks plausible. Continuing should work if the model class is available.


In [4]:
# Run this cell if the version check above warns, then restart the kernel.
# The qwen35 env uses Transformers dev, which currently requires huggingface_hub>=1.5,<2.
if needs_update:
    import subprocess
    import sys

    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-U",
        "git+https://github.com/huggingface/transformers.git",
        "huggingface_hub>=1.5.0,<2.0",
        "accelerate",
        "safetensors",
        "hf_xet",
    ])
    print("Install/update complete. Restart the kernel, then rerun from the top.")
else:
    print("No package update needed.")


No package update needed.


## Load Across GPUs 3 And 4

With `CUDA_VISIBLE_DEVICES=3,4`, `device_map="auto"` can shard the model across the two visible GPUs. The `max_memory` keys use logical device ids `0` and `1`, not physical ids `3` and `4`.


In [5]:
import torch
from transformers import AutoModelForMultimodalLM, AutoProcessor

print("CUDA available:", torch.cuda.is_available())
print("Visible CUDA device count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    total_gib = props.total_memory / 1024**3
    print(f"logical cuda:{i} -> {props.name}, {total_gib:.1f} GiB")

processor = AutoProcessor.from_pretrained(
    MODEL_DIR,
    trust_remote_code=True,
    local_files_only=True,
)

model_kwargs = {
    "trust_remote_code": True,
    "local_files_only": True,
    "low_cpu_mem_usage": True,
}

if torch.cuda.is_available():
    model_kwargs.update(
        torch_dtype=torch.bfloat16,
        device_map="auto",
        # Adjust these if GPUs 3/4 have less or more free memory.
        max_memory={0: "22GiB", 1: "22GiB", "cpu": "80GiB"},
    )
else:
    model_kwargs.update(torch_dtype=torch.float32)

model = AutoModelForMultimodalLM.from_pretrained(MODEL_DIR, **model_kwargs)
model.eval()

print("Device map:")
print(getattr(model, "hf_device_map", "single-device/no device map"))


/home/jorge/miniconda3/envs/qwen35/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


CUDA available: True
Visible CUDA device count: 2
logical cuda:0 -> NVIDIA GeForce RTX 3090, 23.6 GiB
logical cuda:1 -> NVIDIA GeForce RTX 3090, 23.6 GiB


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d
Loading weights: 100%|██████████| 760/760 [00:04<00:00, 153.77it/s]


Device map:
{'model.visual': 0, 'model.language_model.embed_tokens': 0, 'model.language_model.layers.0': 0, 'model.language_model.layers.1': 0, 'model.language_model.layers.2': 0, 'model.language_model.layers.3': 0, 'model.language_model.layers.4': 0, 'model.language_model.layers.5': 0, 'model.language_model.layers.6': 0, 'model.language_model.layers.7': 0, 'model.language_model.layers.8': 0, 'model.language_model.layers.9': 0, 'model.language_model.layers.10': 0, 'model.language_model.layers.11': 1, 'model.language_model.layers.12': 1, 'model.language_model.layers.13': 1, 'model.language_model.layers.14': 1, 'model.language_model.layers.15': 1, 'model.language_model.layers.16': 1, 'model.language_model.layers.17': 1, 'model.language_model.layers.18': 1, 'model.language_model.layers.19': 1, 'model.language_model.layers.20': 1, 'model.language_model.layers.21': 1, 'model.language_model.layers.22': 1, 'model.language_model.layers.23': 1, 'model.language_model.layers.24': 1, 'model.langua

## Text Generation Smoke Test


In [6]:
messages = [
    {
        "role": "system",
        "content": [{"type": "text", "text": "You are a careful assistant for generating child-directed language data."}],
    },
    {
        "role": "user",
        "content": [
            {
                "type": "text",
                "text": "Write 8 short, concrete BabyLM-style sentences about a child finding a red ball. One sentence per line.",
            }
        ],
    },
]

inputs = processor.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt",
)

# For a sharded Accelerate model, place inputs on the first parameter's device.
input_device = next(model.parameters()).device
inputs = {k: v.to(input_device) if hasattr(v, "to") else v for k, v in inputs.items()}

with torch.inference_mode():
    generated_ids = model.generate(
        **inputs,
        max_new_tokens=160,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
    )

new_tokens = generated_ids[:, inputs["input_ids"].shape[-1]:]
text = processor.batch_decode(new_tokens, skip_special_tokens=True)[0]
print(text.strip())


[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


Thinking Process:

1.  **Analyze the Request:**
    *   **Topic:** A child finding a red ball.
    *   **Style:** BabyLM-style (short, concrete, simple vocabulary, child-directed language).
    *   **Quantity:** 8 sentences.
    *   **Format:** One sentence per line.

2.  **Understand "BabyLM-style":**
    *   BabyLMs (like BabyLM-1, BabyLM-2) are models trained on early language data.
    *   Characteristics: Very simple syntax, high-frequency words, concrete nouns and verbs, present tense, minimal abstraction, often narrative or descriptive.
    *   Sentence length: Short (3-6 words ideally).
    *


## Generation Helpers

Default prompts and Qwen wrapper. The default prompt is intentionally neutral: it constrains output form and factual style without naming the experiment.


In [ ]:
DEFAULT_SYSTEM_PROMPT = """
Write plain, natural English about everyday objects, materials, and visible physical behavior.
Use concrete actions and observations.
Avoid rare technical details, microscopic explanations, benchmark-style formatting, and question-answer format.
Use cautious wording such as often, usually, can, or may when a claim depends on context.
When asked for JSON, return only valid JSON.
""".strip()

JSON_SYSTEM_PROMPT = """
Return only valid JSON that follows the requested schema.
Use ordinary everyday knowledge and visible physical behavior.
Avoid rare technical details, microscopic explanations, benchmark-style formatting, and question-answer format.
Use cautious wording when claims depend on context.
""".strip()

PROSE_SYSTEM_PROMPT = """
Write plain, natural prose about everyday objects, materials, and visible physical behavior.
Use concrete actions and observations.
Do not use labels, bullets, question-answer format, or benchmark-style wording.
Avoid rare technical details and microscopic explanations.
""".strip()

EDITOR_SYSTEM_PROMPT = """
Edit for naturalness, concreteness, and everyday plausibility.
Remove benchmark-like formatting, over-technical claims, rare facts, and overly absolute wording.
Keep the intended material, object, event, and behavior intact.
When asked for JSON, return only valid JSON.
""".strip()


def qwen_generate(
    prompt: str,
    *,
    system_prompt: str | None = None,
    max_new_tokens: int = 2500,
    temperature: float = 0.7,
    top_p: float = 0.9,
) -> str:
    messages = [
        {"role": "system", "content": [{"type": "text", "text": system_prompt or DEFAULT_SYSTEM_PROMPT}]},
        {"role": "user", "content": [{"type": "text", "text": prompt}]},
    ]
    inputs = processor.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
        enable_thinking=False,
    )
    input_device = next(model.parameters()).device
    inputs = {k: v.to(input_device) if hasattr(v, "to") else v for k, v in inputs.items()}
    with torch.inference_mode():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=temperature > 0,
            temperature=temperature,
            top_p=top_p,
        )
    new_tokens = generated_ids[:, inputs["input_ids"].shape[-1]:]
    return processor.batch_decode(new_tokens, skip_special_tokens=True)[0].strip()

print("qwen_generate is ready.")


## Load Ontologies

Load curated material classes and object cards from `configs/material_world`. Discourse styles are inline for now, but this cell will use `discourse_styles.json` automatically if that file is added later.


In [ ]:
import json
import re
from pathlib import Path


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    candidates = [start, *start.parents, Path("/home/jorge/babylm-baby-steps")]
    for path in candidates:
        if (path / "pyproject.toml").exists() and (path / "configs").exists():
            return path
    raise FileNotFoundError("Could not find the babylm-baby-steps repo root.")


def load_json(path: Path):
    with path.open() as f:
        return json.load(f)


REPO_ROOT = find_repo_root()
MATERIAL_WORLD_DIR = REPO_ROOT / "configs" / "material_world"
MATERIAL_CLASSES_PATH = MATERIAL_WORLD_DIR / "material_classes.json"
MATERIAL_OBJECTS_PATH = MATERIAL_WORLD_DIR / "material_objects.json"
DISCOURSE_STYLES_PATH = MATERIAL_WORLD_DIR / "discourse_styles.json"

material_class_config = load_json(MATERIAL_CLASSES_PATH)
material_object_config = load_json(MATERIAL_OBJECTS_PATH)

DEFAULT_DISCOURSE_STYLE_CONFIG = {
    "version": "material_world_discourse_styles_inline_v0",
    "description": "Inline starter discourse styles for material-world pilot generation.",
    "discourse_styles": [
        {
            "style_id": "everyday_narrative",
            "style_name": "Everyday narrative",
            "target_length_tokens": [128, 256],
            "prompt_hint": "Write a short everyday scene with concrete actions and observations.",
            "avoid": ["labels", "bullet lists", "question-answer format"],
        },
        {
            "style_id": "comparison_paragraph",
            "style_name": "Comparison paragraph",
            "target_length_tokens": [128, 256],
            "prompt_hint": "Compare how two materials change the same object or event in one natural paragraph.",
            "avoid": ["tables", "numbered lists", "test-like wording"],
        },
        {
            "style_id": "child_science_explanation",
            "style_name": "Child science explanation",
            "target_length_tokens": [128, 256],
            "prompt_hint": "Explain the visible behavior in simple language without sounding like a quiz.",
            "avoid": ["microscopic mechanisms", "precise measurements", "advanced terminology"],
        },
        {
            "style_id": "repair_craft_note",
            "style_name": "Repair/craft note",
            "target_length_tokens": [128, 256],
            "prompt_hint": "Write like a practical note about making, fixing, washing, or choosing an object.",
            "avoid": ["instructions with numbered steps", "overly technical advice"],
        },
        {
            "style_id": "observation_log",
            "style_name": "Observation log",
            "target_length_tokens": [128, 256],
            "prompt_hint": "Write a plain observation of what happened before, during, and after an event.",
            "avoid": ["lab-report labels", "measurements", "formal hypothesis language"],
        },
        {
            "style_id": "short_dialogue",
            "style_name": "Short dialogue",
            "target_length_tokens": [128, 256],
            "prompt_hint": "Write a brief natural conversation where the material behavior is noticed in context.",
            "avoid": ["quiz questions", "lecturing", "speaker labels like C1 or T1"],
        },
        {
            "style_id": "simple_encyclopedia_paragraph",
            "style_name": "Simple encyclopedia paragraph",
            "target_length_tokens": [128, 256],
            "prompt_hint": "Write a neutral paragraph with ordinary examples and cautious claims.",
            "avoid": ["rare facts", "long lists", "precise scientific mechanisms"],
        },
        {
            "style_id": "application_tradeoff_paragraph",
            "style_name": "Application/tradeoff paragraph",
            "target_length_tokens": [128, 256],
            "prompt_hint": "Explain why one material works better or worse for an object in a normal use case.",
            "avoid": ["engineering jargon", "absolute claims", "benchmark-style contrasts"],
        },
    ],
}

discourse_style_config = load_json(DISCOURSE_STYLES_PATH) if DISCOURSE_STYLES_PATH.exists() else DEFAULT_DISCOURSE_STYLE_CONFIG

material_classes = material_class_config["material_classes"]
material_classes_by_id = {item["class_id"]: item for item in material_classes}
material_examples = {item["class_id"]: item["examples"] for item in material_classes}

material_objects = material_object_config["objects"]
material_objects_by_id = {item["object_id"]: item for item in material_objects}
object_candidate_materials = {item["object_id"]: item["candidate_materials"] for item in material_objects}

discourse_styles = discourse_style_config.get("discourse_styles", discourse_style_config.get("styles", []))
discourse_styles_by_id = {item["style_id"]: item for item in discourse_styles}

print(f"Loaded {len(material_classes)} material classes from {MATERIAL_CLASSES_PATH}")
print(f"Loaded {len(material_objects)} object cards from {MATERIAL_OBJECTS_PATH}")
print(f"Loaded {len(discourse_styles)} discourse styles")
{
    "material_classes": [item["class_name"] for item in material_classes],
    "sample_objects": [item["object_id"] for item in material_objects[:8]],
    "discourse_styles": [item["style_id"] for item in discourse_styles],
}


## JSON Helpers

Qwen sometimes returns fenced JSON. These helpers extract, validate, and optionally save generated artifacts.


In [ ]:
def extract_json(text: str):
    """Extract the first valid JSON object or array from a model response."""
    text = text.strip()
    candidates = [match.group(1).strip() for match in re.finditer(r"```(?:json)?\s*(.*?)```", text, flags=re.S | re.I)]
    candidates.append(text)

    decoder = json.JSONDecoder()
    errors = []
    for candidate in candidates:
        try:
            return json.loads(candidate)
        except json.JSONDecodeError as exc:
            errors.append(str(exc))

        for start, char in enumerate(candidate):
            if char not in "[{":
                continue
            try:
                obj, _ = decoder.raw_decode(candidate[start:])
                return obj
            except json.JSONDecodeError:
                continue

    raise ValueError("No valid JSON object or array found in model response. Last errors: " + " | ".join(errors[-3:]))


def require_json_list(value, name: str = "value"):
    if not isinstance(value, list):
        raise TypeError(f"{name} should be a JSON list, got {type(value).__name__}")
    return value


def write_json(path: Path, data) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)
        f.write("\n")


def write_jsonl(path: Path, records) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w") as f:
        for record in records:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")

print("JSON helpers are ready.")


## Pilot Controls

Keep these limits small while inspecting quality. Increase them only after the generated plans and prose look good.


In [ ]:
PILOT_OUTPUT_DIR = REPO_ROOT / "data" / "material_world" / "pilot_v0"

PILOT_MATERIAL_CLASS_LIMIT = 1
PILOT_OBJECT_LIMIT = 1
PILOT_STYLE_LIMIT = 2
PILOT_SCENARIO_LIMIT = 2

selected_material_classes = material_classes[:PILOT_MATERIAL_CLASS_LIMIT]
selected_material_objects = material_objects[:PILOT_OBJECT_LIMIT]
selected_discourse_styles = discourse_styles[:PILOT_STYLE_LIMIT]

print(f"Pilot material classes: {[item['class_id'] for item in selected_material_classes]}")
print(f"Pilot objects: {[item['object_id'] for item in selected_material_objects]}")
print(f"Pilot styles: {[item['style_id'] for item in selected_discourse_styles]}")
print(f"Pilot output dir: {PILOT_OUTPUT_DIR}")


## Generate Material Cards

Generate a tiny batch of structured material cards from the curated material classes. The variable `example_material_cards` holds parsed JSON for later cells.


In [ ]:
material_card_prompt = f"""
Create JSON material cards for the material classes below.

Requirements:
- Return a JSON array only.
- Create one card per material class.
- Preserve material_class_id and material_class_name.
- Use list fields, not comma-separated strings.
- Use ordinary everyday knowledge and visible physical behavior.
- Avoid rare technical details, exact measurements, microscopic explanations, and dubious claims.
- Use cautious wording such as often, usually, can, and may when needed.

Each card should use this schema:
{{
  "material_class_id": "...",
  "material_class_name": "...",
  "everyday_examples": ["..."],
  "core_properties": ["..."],
  "common_dynamics": ["..."],
  "unlikely_or_context_limited_dynamics": ["..."],
  "everyday_objects": ["..."],
  "caveats": ["..."],
  "simple_observations": ["..."]
}}

Material classes:
{json.dumps(selected_material_classes, indent=2)}
""".strip()

raw_material_cards = qwen_generate(
    material_card_prompt,
    system_prompt=JSON_SYSTEM_PROMPT,
    max_new_tokens=3500,
    temperature=0.2,
)
print(raw_material_cards)

example_material_cards = require_json_list(extract_json(raw_material_cards), "example_material_cards")
print(f"Parsed {len(example_material_cards)} material cards.")


## Generate Scenario Plans

Create structured object-material scenario plans before rendering prose. This keeps the material, object, event, property, dynamic, style, and contrast metadata available for filtering and balancing.


In [ ]:
material_context = example_material_cards if "example_material_cards" in globals() else selected_material_classes

scenario_plan_prompt = f"""
Create a small JSON array of object-material scenario plans.

Requirements:
- Return JSON only.
- Use the selected object cards and material context below.
- Vary materials while keeping some objects or events comparable.
- Prefer visible everyday dynamics such as cracking, denting, folding, soaking, compressing, bouncing, dripping, scratching, warping, softening, and crumbling.
- Avoid benchmark-style examples, question-answer format, labels like C1/T1, rare facts, and microscopic mechanisms.
- Include negative_reporting=true only when the prose should mention what would not happen or what another material would do differently.

Each scenario should use this schema:
{{
  "scenario_id": "...",
  "material_class_id": "...",
  "material": "...",
  "object_id": "...",
  "object_name": "...",
  "event": "...",
  "visible_property": "...",
  "dynamics": ["..."],
  "expected_behavior": "...",
  "contrast_material": "...",
  "contrast_behavior": "...",
  "discourse_style_id": "...",
  "negative_reporting": true
}}

Create at most {PILOT_SCENARIO_LIMIT} scenarios.

Material context:
{json.dumps(material_context, indent=2)}

Object cards:
{json.dumps(selected_material_objects, indent=2)}

Discourse styles:
{json.dumps(selected_discourse_styles, indent=2)}
""".strip()

raw_scenario_plans = qwen_generate(
    scenario_plan_prompt,
    system_prompt=JSON_SYSTEM_PROMPT,
    max_new_tokens=4000,
    temperature=0.3,
)
print(raw_scenario_plans)

example_scenario_plans = require_json_list(extract_json(raw_scenario_plans), "example_scenario_plans")
print(f"Parsed {len(example_scenario_plans)} scenario plans.")


## Render Natural Prose

Render scenario plans into natural text while keeping metadata outside the final training text.


In [ ]:
def scenario_dynamics(plan: dict) -> list[str]:
    dynamics = plan.get("dynamics", [])
    if isinstance(dynamics, str):
        dynamics = [dynamics]
    dynamics = [str(item) for item in dynamics if item]
    if dynamics:
        return dynamics
    fallback = plan.get("visible_property") or plan.get("expected_behavior")
    return [str(fallback)] if fallback else []


def audit_metadata_from_plan(plan: dict) -> dict:
    return {
        "style_id": plan.get("discourse_style_id"),
        "material": plan.get("material"),
        "material_class": plan.get("material_class_id"),
        "object": plan.get("object_name") or plan.get("object_id"),
        "object_id": plan.get("object_id"),
        "event": plan.get("event"),
        "visible_property": plan.get("visible_property"),
        "dynamics": scenario_dynamics(plan),
        "contains_negative_reporting": bool(plan.get("negative_reporting", False)),
        "contrast_material": plan.get("contrast_material"),
        "contrast_behavior": plan.get("contrast_behavior"),
    }


render_prompt = f"""
Render these object-material scenario plans into natural prose.

Requirements:
- Return JSON only.
- Return one object per scenario with scenario_id, discourse_style_id, and text.
- The text should sound like ordinary prose, not a benchmark item.
- Do not use labels, bullets, tables, question-answer format, or template-like wording.
- Keep the material-conditioned behavior from the plan.
- Use cautious wording when the behavior depends on thickness, heat, force, age, or context.
- Each text should be roughly 70 to 150 words.

Scenario plans:
{json.dumps(example_scenario_plans[:PILOT_SCENARIO_LIMIT], indent=2)}
""".strip()

raw_rendered_examples = qwen_generate(
    render_prompt,
    system_prompt=JSON_SYSTEM_PROMPT,
    max_new_tokens=4500,
    temperature=0.6,
)
print(raw_rendered_examples)

example_rendered_examples = require_json_list(extract_json(raw_rendered_examples), "example_rendered_examples")
rendered_by_id = {item.get("scenario_id") or item.get("id"): item for item in example_rendered_examples}

prose_draft_examples = []
for index, plan in enumerate(example_scenario_plans[:PILOT_SCENARIO_LIMIT]):
    scenario_id = plan.get("scenario_id", f"scenario_{index:04d}")
    rendered = rendered_by_id.get(scenario_id, example_rendered_examples[index] if index < len(example_rendered_examples) else {})
    text = str(rendered.get("text", "")).strip()
    prose_draft_examples.append(
        {
            "id": scenario_id,
            "text": text,
            "metadata": audit_metadata_from_plan(plan),
            "scenario_plan": plan,
        }
    )

pilot_train_text = "\n\n".join(item["text"] for item in prose_draft_examples if item.get("text"))
print(f"Parsed {len(example_rendered_examples)} rendered examples.")
print(f"Built {len(prose_draft_examples)} audit-ready draft examples.")
print(pilot_train_text[:2000])


## Lightweight Editor / Check Pass

Use this pass to revise or drop prose that is too technical, too broad, factually dubious, impossible without framing, or too template-like.


In [ ]:
check_prompt = f"""
Review these rendered examples.

For each example:
- Keep it if it is natural, concrete, everyday, and plausible.
- Revise it if it is too technical, too broad, too absolute, repetitive, or benchmark-like.
- Drop it only if it is factually dubious, impossible without pretend framing, or not about visible material behavior.
- Preserve the intended material, object, event, and behavior when revising.

Return JSON only. Use this schema for each item:
{{
  "id": "...",
  "status": "keep|revise|drop",
  "text": "final text if kept or revised, empty string if dropped",
  "notes": ["short reason"]
}}

Rendered examples:
{json.dumps(prose_draft_examples, indent=2)}
""".strip()

raw_checked_examples = qwen_generate(
    check_prompt,
    system_prompt=EDITOR_SYSTEM_PROMPT,
    max_new_tokens=4500,
    temperature=0.2,
)
print(raw_checked_examples)

checked_examples = require_json_list(extract_json(raw_checked_examples), "checked_examples")
checked_by_id = {item.get("id") or item.get("scenario_id"): item for item in checked_examples}

prose_checked_examples = []
for draft in prose_draft_examples:
    checked = checked_by_id.get(draft["id"], {})
    status = checked.get("status", "keep")
    text = str(checked.get("text") or draft.get("text", "")).strip()
    row = dict(draft)
    row["text"] = "" if status == "drop" else text
    row["metadata"] = dict(draft.get("metadata", {}))
    row["metadata"]["checker_status"] = status
    row["metadata"]["checker_notes"] = checked.get("notes", [])
    row["checker_output"] = checked
    prose_checked_examples.append(row)

accepted_examples = [item for item in prose_checked_examples if item.get("text") and item["metadata"].get("checker_status") != "drop"]
checked_train_text = "\n\n".join(item["text"].strip() for item in accepted_examples)
print(f"Accepted {len(accepted_examples)} / {len(prose_checked_examples)} examples.")
print(checked_train_text[:2000])


## Export Pilot Files

Run this only after inspecting the generated examples. The final training text export contains prose only; metadata stays in JSON/JSONL files.


In [ ]:
if "example_scenario_plans" not in globals() or "accepted_examples" not in globals():
    raise RuntimeError("Run the scenario, rendering, and check-pass cells before exporting.")

PILOT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if "example_material_cards" in globals():
    write_json(PILOT_OUTPUT_DIR / "material_cards.json", example_material_cards)

write_json(PILOT_OUTPUT_DIR / "scenario_plans.json", example_scenario_plans)
write_jsonl(PILOT_OUTPUT_DIR / "rendered_examples.jsonl", example_rendered_examples)
write_jsonl(PILOT_OUTPUT_DIR / "prose_draft.jsonl", prose_draft_examples)
write_jsonl(PILOT_OUTPUT_DIR / "prose_checked_all.jsonl", prose_checked_examples)
write_jsonl(PILOT_OUTPUT_DIR / "prose_checked.jsonl", accepted_examples)
write_jsonl(PILOT_OUTPUT_DIR / "checked_examples.jsonl", accepted_examples)
(PILOT_OUTPUT_DIR / "train_checked.txt").write_text(checked_train_text.strip() + "\n")
(PILOT_OUTPUT_DIR / "train.txt").write_text(checked_train_text.strip() + "\n")

print(f"Wrote pilot files to {PILOT_OUTPUT_DIR}")
print(f"Accepted examples exported for audit: {len(accepted_examples)}")
print(f"Approx whitespace tokens in train_checked.txt: {len(checked_train_text.split())}")


## Audit Checked Prose

Run the programmatic quality/diversity audit on the metadata-rich checked prose. This does not call Qwen; it adds `audit_flags` and writes a corpus-level report.


In [ ]:
import sys

SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from babysteps.material_world.audit import audit_jsonl_file, read_jsonl

PROSE_CHECKED_JSONL = PILOT_OUTPUT_DIR / "prose_checked.jsonl"
AUDIT_OUTPUT_JSONL = PILOT_OUTPUT_DIR / "examples_audited.jsonl"
AUDIT_REPORT_JSON = PILOT_OUTPUT_DIR / "audit_report.json"
AUDIT_DROP_IDS = PILOT_OUTPUT_DIR / "drop_ids.txt"

initial_audit_result = audit_jsonl_file(
    PROSE_CHECKED_JSONL,
    output_jsonl=AUDIT_OUTPUT_JSONL,
    report_json=AUDIT_REPORT_JSON,
    drop_ids_output=AUDIT_DROP_IDS,
)

audited_examples = read_jsonl(AUDIT_OUTPUT_JSONL)
print(f"Audited {initial_audit_result.report['total_examples']} examples.")
print(f"Warnings: {initial_audit_result.report['examples_warned']} | Drops: {initial_audit_result.report['examples_dropped']}")
print("Recommendations:")
for recommendation in initial_audit_result.report["recommendations"]:
    print("-", recommendation)


## Audit-Guided Qwen Edit Pass

Use audit flags as targeted edit instructions. This edits only warning rows with row-level issues and leaves hard drops out of the edit set. Batch-level balance warnings should usually guide the next generation batch instead of rewriting every row.


In [ ]:
ROW_LEVEL_AUDIT_EDIT_FLAGS = {
    "repetitive_opening_5",
    "repetitive_opening_8",
    "many_connectives",
    "because_overuse",
    "contrast_phrase_overuse",
    "too_short",
    "too_long",
    "template_phrase_overuse",
    "repeated_first_12_words",
    "repeated_8gram",
    "near_duplicate_candidate",
}

BATCH_LEVEL_AUDIT_FLAGS = {
    "material_dynamic_collapse",
}

AUDIT_EDIT_LIMIT = 20


def row_level_flags(row: dict) -> list[dict]:
    return [
        flag
        for flag in row.get("audit_flags", [])
        if flag.get("code") in ROW_LEVEL_AUDIT_EDIT_FLAGS and flag.get("severity") != "hard_drop"
    ]


def audit_edit_prompt(row: dict, flags: list[dict]) -> str:
    return f"""
Revise this prose example using the audit notes.

Rules:
- Keep the same material, object, event, and visible material behavior.
- Keep ordinary natural prose.
- Fix only the audit issues listed below.
- Do not add labels, bullets, question-answer format, or benchmark wording.
- Do not add rare technical details or microscopic explanations.
- Use cautious wording when needed.
- Return JSON only with fields: id, text.

Metadata:
{json.dumps(row.get('metadata', {}), indent=2)}

Audit flags to fix:
{json.dumps(flags, indent=2)}

Original text:
{row.get('text', '')}
""".strip()

rows_to_audit_edit = [
    row
    for row in audited_examples
    if row.get("audit_status") == "warn" and row_level_flags(row)
]
rows_to_audit_edit = rows_to_audit_edit[:AUDIT_EDIT_LIMIT]
print(f"Rows selected for audit-guided editing: {len(rows_to_audit_edit)}")

edited_by_id = {}
for row in rows_to_audit_edit:
    flags = row_level_flags(row)
    raw_edit = qwen_generate(
        audit_edit_prompt(row, flags),
        system_prompt=EDITOR_SYSTEM_PROMPT,
        max_new_tokens=900,
        temperature=0.25,
    )
    print(raw_edit)
    edited = extract_json(raw_edit)
    edited_by_id[str(edited.get("id", row["id"]))] = edited

prose_audit_edited_examples = []
for row in audited_examples:
    if row.get("audit_status") == "drop":
        continue
    edited = edited_by_id.get(str(row["id"]))
    next_row = dict(row)
    next_row.pop("audit_flags", None)
    next_row.pop("audit_status", None)
    next_row["metadata"] = dict(row.get("metadata", {}))
    if edited and edited.get("text"):
        next_row["text"] = str(edited["text"]).strip()
        next_row["metadata"]["audit_edited"] = True
        next_row["metadata"]["audit_flags_before_edit"] = row.get("audit_flags", [])
    else:
        next_row["metadata"]["audit_edited"] = False
    prose_audit_edited_examples.append(next_row)

PROSE_AUDIT_EDITED_JSONL = PILOT_OUTPUT_DIR / "prose_audit_edited.jsonl"
write_jsonl(PROSE_AUDIT_EDITED_JSONL, prose_audit_edited_examples)
print(f"Wrote {len(prose_audit_edited_examples)} audit-edited examples to {PROSE_AUDIT_EDITED_JSONL}")


## Re-Audit And Final Export

Re-run the programmatic audit after the targeted edit pass. The final training text excludes hard drops and keeps prose only.


In [ ]:
FINAL_AUDIT_OUTPUT_JSONL = PILOT_OUTPUT_DIR / "examples_reaudited.jsonl"
FINAL_AUDIT_REPORT_JSON = PILOT_OUTPUT_DIR / "audit_report_after_edit.json"
FINAL_DROP_IDS = PILOT_OUTPUT_DIR / "drop_ids_after_edit.txt"
FINAL_TRAIN_TXT = PILOT_OUTPUT_DIR / "train_audited.txt"

final_audit_result = audit_jsonl_file(
    PROSE_AUDIT_EDITED_JSONL,
    output_jsonl=FINAL_AUDIT_OUTPUT_JSONL,
    report_json=FINAL_AUDIT_REPORT_JSON,
    drop_ids_output=FINAL_DROP_IDS,
)

reaudited_examples = read_jsonl(FINAL_AUDIT_OUTPUT_JSONL)
final_training_rows = [row for row in reaudited_examples if row.get("audit_status") != "drop" and row.get("text")]
final_train_text = "\n\n".join(row["text"].strip() for row in final_training_rows)
FINAL_TRAIN_TXT.write_text(final_train_text.strip() + "\n")

print(f"Final kept examples: {len(final_training_rows)} / {len(reaudited_examples)}")
print(f"Approx whitespace tokens in train_audited.txt: {len(final_train_text.split())}")
print("Post-edit recommendations:")
for recommendation in final_audit_result.report["recommendations"]:
    print("-", recommendation)
print(f"Final prose-only export: {FINAL_TRAIN_TXT}")


## Scratch Prompts

Older exploratory prompts live here so they do not interrupt the main pilot flow.


In [9]:
print(qwen_generate("List everyday material dynamics that are common, visible, and useful for a child or small language model to learn from text. Avoid microscopic physics, rare engineering terms, precise scientific mechanisms, and niche phenomena. Focus on things like bending, folding, tearing, soaking, dripping, denting, cracking, bouncing, compressing, stretching, melting, drying, sticking, sliding, pouring, piling, crumpling, and sagging. For each event, give: 1. common materials involved, 2. what happens, 3. what property explains it, 4. one everyday scenario, 5. one natural negative contrast."))

[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


Here is a list of everyday material dynamics suitable for learning from text, focusing on observable actions and simple properties.

| Event | Common Materials | What Happens | Explaining Property | Everyday Scenario | Natural Negative Contrast |
| :--- | :--- | :--- | :--- :--- | :--- |
| **Bending** | Paper, Plastic, Wood | An object curves under force without breaking. | Flexibility | Bending a metal spoon to open a jar. | Glass (shatters instead of bending). |
| **Folding** | Paper, Fabric, Aluminum Foil | One part of a material overlaps another along a crease. | Plasticity | Folding a napkin into a triangle for a party. | Tearing (the paper rips instead of creasing). |
| **Tearing** | Paper, Cloth, Dry Leaves | A material splits apart when pulled too hard. | Brittleness | Tearing a page out of a notebook. | Gluing (the pieces stick together instead of separating). |
| **Soaking** | Sponge, Towel, Soil | A dry material absorbs liquid and becomes heavy/wet. | Porosity | Soaking a ra

In [10]:
print(qwen_generate("Assume you design and engineer skateboards. Describe the utility of a skateboard, and how would the dynamics change of a skateboard if it was made of different materials, e.g. plastic, metal, styrofoam, cloth, etc. Give one scenario with these different products "))

[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


### Utility of a Skateboard
A skateboard serves as a mobile platform for personal transportation, recreation, and sport. Its primary utility relies on converting human kinetic energy into forward momentum and allowing the rider to manipulate that motion through tricks, balancing, and terrain negotiation. The deck acts as a lever, while the wheels and trucks provide suspension, traction, and directional stability.

### Material Dynamics Analysis
The material composition of the deck fundamentally alters the board's mechanical properties, specifically **stiffness (flex)**, **weight**, and **energy dissipation**.

*   **Plastic (e.g., ABS or Polycarbonate):**
    *   *Dynamics:* Offers moderate stiffness with high impact resistance. It provides a consistent, predictable feel similar to traditional wood but is generally heavier.
    *   *Performance:* Good for beginners due to durability; however, excessive flex can reduce the "pop" (vertical snap) required for ollies.
*   **Metal (e.g., Al